# picoGPT：亲眼看见一次 GPT-2 推理

从一小段文本出发，沿着本仓库 `gpt2.py` 的计算顺序，观察 **token → embedding → Q/K/V → attention → MLP → logits → 下一个 token**。

这里使用仓库已下载的 **GPT-2 124M 真实权重**。为了看清矩阵，只完整展示第一层的第一个 attention head；其余 head 和层照常计算。每格可从上往下运行。

## 1. 准备

使用本项目的 Python 环境作为 Kernel。Notebook 放在仓库根目录；权重必须已存在于 `models/124M/`。这份 Notebook 只读取本地文件，不会下载模型。

In [1]:
from pathlib import Path
from html import escape
import numpy as np
from IPython.display import HTML, display

from gpt2 import attention, ffn, gpt2, layer_norm, linear, softmax, transformer_block
from utils import load_encoder_hparams_and_params

assert Path('gpt2.py').is_file(), '请把工作目录设为 picoGPT 仓库根目录'
assert Path('models/124M/model.ckpt.index').is_file(), '缺少本地 GPT-2 124M 权重'
np.set_printoptions(precision=3, suppress=True)

def show_matrix(values, row_labels, col_labels, title, masked=False):
    values = np.asarray(values)
    assert values.shape == (len(row_labels), len(col_labels))
    finite = values[values > -1e9] if masked else values.ravel()
    lo, hi = float(finite.min()), float(finite.max())
    span = max(hi - lo, 1e-12)
    head = '<tr><th></th>' + ''.join(f'<th>{escape(str(c))}</th>' for c in col_labels) + '</tr>'
    rows = []
    for label, row in zip(row_labels, values):
        cells = []
        for value in row:
            if masked and value <= -1e9:
                cells.append('<td style="background:#eee;color:#777">×</td>')
            else:
                intensity = (float(value) - lo) / span
                shade = int(245 - 155 * intensity)
                cells.append(f'<td style="background:rgb({shade},{shade + 7},{255})">{value:.3f}</td>')
        rows.append('<tr><th>' + escape(str(label)) + '</th>' + ''.join(cells) + '</tr>')
    style = '<style>table.attn{border-collapse:collapse;font:14px sans-serif}table.attn td,table.attn th{padding:7px 10px;border:1px solid #bbb;text-align:center}</style>'
    display(HTML(style + f'<p><b>{escape(title)}</b></p><table class="attn">' + head + ''.join(rows) + '</table>'))

def preview(name, array, rows=2, cols=8):
    array = np.asarray(array)
    print(f'{name}: shape={array.shape}, dtype={array.dtype}')
    if array.ndim == 2:
        print(array[:rows, :cols], f'  ← 前 {rows} 行 × 前 {cols} 列')
    else:
        print(array[:cols])


### 读取真实权重

`utils.py` 将本地 TensorFlow checkpoint 转成 NumPy 数组。首次执行会占用约数百 MB 内存；输出只显示模型规格。

In [2]:
encoder, hparams, params = load_encoder_hparams_and_params('124M', 'models')
print({key: hparams[key] for key in ('n_vocab', 'n_ctx', 'n_embd', 'n_head', 'n_layer')})
print('head_dim =', hparams['n_embd'] // hparams['n_head'])


{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
head_dim = 64


## 2. 文本变成 token ID

GPT-2 的 tokenizer 会把字符串切成词片段。**模型接收的是整数 ID**，下面单独 decode 每个 ID，只为让人看懂。

In [4]:
prompt = 'The capital of France is'
input_ids = encoder.encode(prompt)
tokens = [encoder.decode([int(token_id)]) for token_id in input_ids]
print('原文:', repr(prompt))
print('token IDs:', input_ids)
print('对应片段:', [repr(token) for token in tokens])
print('序列长度 N =', len(input_ids))


原文: 'The capital of France is'
token IDs: [464, 3139, 286, 4881, 318]
对应片段: ["'The'", "' capital'", "' of'", "' France'", "' is'"]
序列长度 N = 5


## 3. Token embedding + position embedding

对应 `gpt2.py` 第 75 行。`wte[id]` 给每个 token 查一行向量；`wpe[position]` 给每个位置查一行向量。两者相加得到 `X`，形状为 `[N, 768]`。

In [5]:
token_embedding = params['wte'][input_ids]
position_embedding = params['wpe'][np.arange(len(input_ids))]
x0 = token_embedding + position_embedding
preview('token embedding', token_embedding)
preview('position embedding', position_embedding)
preview('X = 两者相加', x0)
assert np.allclose(x0, token_embedding + position_embedding)


token embedding: shape=(5, 768), dtype=float32
[[-0.069 -0.02   0.064 -0.062 -0.114 -0.062 -0.217 -0.03 ]
 [ 0.057 -0.007  0.091  0.064 -0.029  0.058 -0.364  0.063]]   ← 前 2 行 × 前 8 列
position embedding: shape=(5, 768), dtype=float32
[[-0.019 -0.197  0.004  0.011  0.064 -0.105  0.037 -0.168]
 [ 0.024 -0.054 -0.095 -0.013 -0.01   0.031  0.051  0.185]]   ← 前 2 行 × 前 8 列
X = 两者相加: shape=(5, 768), dtype=float32
[[-0.087 -0.218  0.068 -0.051 -0.05  -0.167 -0.18  -0.198]
 [ 0.081 -0.061 -0.003  0.051 -0.039  0.089 -0.313  0.248]]   ← 前 2 行 × 前 8 列


## 4. 第一层：LayerNorm → Q/K/V

对应 `transformer_block()` 与 `mha()`。GPT-2 先做 LayerNorm，再用**一次线性投影**得到拼在一起的 Q/K/V，随后拆成 12 个 head。下方重点看第 0 个 head。

In [5]:
block0 = params['blocks'][0]
x_norm = layer_norm(x0, **block0['ln_1'])
qkv_joined = linear(x_norm, **block0['attn']['c_attn'])
q_all, k_all, v_all = np.split(qkv_joined, 3, axis=-1)
q_heads = np.split(q_all, hparams['n_head'], axis=-1)
k_heads = np.split(k_all, hparams['n_head'], axis=-1)
v_heads = np.split(v_all, hparams['n_head'], axis=-1)
q, k, v = q_heads[0], k_heads[0], v_heads[0]
for name, value in [('LayerNorm(X)', x_norm), ('QKV 合并', qkv_joined),
                    ('Q（全部 head）', q_all), ('Q（第 0 个 head）', q),
                    ('K（第 0 个 head）', k), ('V（第 0 个 head）', v)]:
    preview(name, value, rows=1, cols=6)


LayerNorm(X): shape=(5, 768), dtype=float32
[[-0.053 -0.078 -0.032 -0.028 -0.04  -0.097]]   ← 前 1 行 × 前 6 列
QKV 合并: shape=(5, 2304), dtype=float32
[[ 0.153 -0.415  0.367 -1.178  0.168  0.663]]   ← 前 1 行 × 前 6 列
Q（全部 head）: shape=(5, 768), dtype=float32
[[ 0.153 -0.415  0.367 -1.178  0.168  0.663]]   ← 前 1 行 × 前 6 列
Q（第 0 个 head）: shape=(5, 64), dtype=float32
[[ 0.153 -0.415  0.367 -1.178  0.168  0.663]]   ← 前 1 行 × 前 6 列
K（第 0 个 head）: shape=(5, 64), dtype=float32
[[-0.942  1.902  0.872 -0.15   0.404  0.208]]   ← 前 1 行 × 前 6 列
V（第 0 个 head）: shape=(5, 64), dtype=float32
[[-0.013 -0.014  0.127 -0.022 -0.208  0.098]]   ← 前 1 行 × 前 6 列


## 5. 第 0 个 head：QKᵀ / √d

每一行是一个 **query token**，每一列是一个 **key token**。数值越高，表示这个 query 对该 key 的未归一化匹配分数越高。矩阵形状是 `[N, N]`。

In [6]:
scores = q @ k.T / np.sqrt(q.shape[-1])
print('Q:', q.shape, 'K:', k.shape, 'scores:', scores.shape)
show_matrix(scores, tokens, tokens, '缩放后的 attention scores（尚未遮挡未来 token）')


Q: (5, 64) K: (5, 64) scores: (5, 5)


,The,capital,of,France,is
The,-0.112,-1.410,-2.180,-1.792,-2.388
capital,0.754,-0.953,-1.679,-2.574,-2.571
of,-0.155,-1.297,-2.391,-1.841,-2.041
France,0.307,-0.916,-1.133,-1.860,-1.555
is,-0.553,-1.943,-2.830,-2.418,-2.570


## 6. 因果掩码：不能偷看未来

对应 `gpt2.py` 第 49 行。上三角位置加上 `-1e10`，softmax 后概率约为 0。表中的 `×` 表示不可见的位置。

In [7]:
causal_mask = (1 - np.tri(len(input_ids), dtype=q.dtype)) * -1e10
masked_scores = scores + causal_mask
show_matrix(masked_scores, tokens, tokens, '加因果掩码后的 scores', masked=True)
upper = np.triu_indices(len(input_ids), k=1)
assert np.all(masked_scores[upper] <= -1e9)


,The,capital,of,France,is
The,-0.112,×,×,×,×
capital,0.754,-0.953,×,×,×
of,-0.155,-1.297,-2.391,×,×
France,0.307,-0.916,-1.133,-1.860,×
is,-0.553,-1.943,-2.830,-2.418,-2.570


## 7. Softmax：每行变成概率

沿**最后一个维度**做 softmax。每行概率之和为 1；最后一行可以看见提示词中的所有 token。这里的概率是 attention 权重，**不是最终的下一个 token 概率**。

In [8]:
attn_probs = softmax(masked_scores)
show_matrix(attn_probs, tokens, tokens, '第 0 个 head 的 attention 权重')
print('每行概率之和:', attn_probs.sum(axis=-1))
print('未来位置的最大概率:', np.triu(attn_probs, k=1).max())
assert np.allclose(attn_probs.sum(axis=-1), 1)


,The,capital,of,France,is
The,1.000,0.000,0.000,0.000,0.000
capital,0.846,0.154,0.000,0.000,0.000
of,0.701,0.224,0.075,0.000,0.000
France,0.608,0.179,0.144,0.070,0.000
is,0.610,0.152,0.063,0.094,0.081


每行概率之和: [1. 1. 1. 1. 1.]
未来位置的最大概率: 0.0


## 8. 用概率对 V 加权求和

`P @ V` 的第 `i` 行是第 `i` 个 token 汇聚之前所有可见 token 的 value 向量。输出重新回到 `[N, 64]`；第 0 个 head 仍只是 12 个 head 中的一个。

In [9]:
head0_output = attn_probs @ v
preview('第 0 个 head 的 P @ V', head0_output, rows=2, cols=8)
print('核对仓库 attention() 的最大差值:', np.max(np.abs(head0_output - attention(q, k, v, causal_mask))))
assert np.allclose(head0_output, attention(q, k, v, causal_mask))


第 0 个 head 的 P @ V: shape=(5, 64), dtype=float32
[[-0.013 -0.014  0.127 -0.022 -0.208  0.098  0.129 -0.087]
 [-0.011  0.005  0.125  0.038 -0.147  0.051  0.027 -0.087]]   ← 前 2 行 × 前 8 列
核对仓库 attention() 的最大差值: 0.0


## 9. 合并所有 head，再加残差

其余 11 个 head 按相同公式计算。沿特征维拼接成 `[N, 768]`，经过 `c_proj`，再加回输入 `X`。这对应 `mha()` 的后半段和 `transformer_block()` 的第一个残差。

In [10]:
all_head_outputs = [attention(qh, kh, vh, causal_mask)
                    for qh, kh, vh in zip(q_heads, k_heads, v_heads)]
all_head_outputs[0] = head0_output
merged_heads = np.hstack(all_head_outputs)
attention_projection = linear(merged_heads, **block0['attn']['c_proj'])
x_after_attention = x0 + attention_projection
preview('12 个 head 拼接', merged_heads, rows=1)
preview('输出投影', attention_projection, rows=1)
preview('残差 X + attention 输出', x_after_attention, rows=1)


12 个 head 拼接: shape=(5, 768), dtype=float32
[[-0.013 -0.014  0.127 -0.022 -0.208  0.098  0.129 -0.087]]   ← 前 1 行 × 前 8 列
输出投影: shape=(5, 768), dtype=float32
[[ 0.209 -0.214  0.738 -0.013  0.007 -0.002 -0.598 -0.434]]   ← 前 1 行 × 前 8 列
残差 X + attention 输出: shape=(5, 768), dtype=float32
[[ 0.122 -0.431  0.806 -0.064 -0.043 -0.17  -0.778 -0.632]]   ← 前 1 行 × 前 8 列


## 10. MLP，再加一次残差

`ffn()` 做 `768 → 3072 → 768`。它**逐 token** 工作，不会像 attention 一样让不同 token 互相通信。第一层最终输出与仓库 `transformer_block()` 逐项核对。

In [11]:
mlp_input = layer_norm(x_after_attention, **block0['ln_2'])
mlp_hidden = linear(mlp_input, **block0['mlp']['c_fc'])
from gpt2 import gelu
mlp_activated = gelu(mlp_hidden)
mlp_output = linear(mlp_activated, **block0['mlp']['c_proj'])
x_after_block0 = x_after_attention + mlp_output
for name, value in [('MLP 输入', mlp_input), ('升维后', mlp_hidden),
                    ('GELU 后', mlp_activated), ('降维后', mlp_output),
                    ('第一层最终输出', x_after_block0)]:
    preview(name, value, rows=1, cols=6)
reference_block0 = transformer_block(x0, **block0, n_head=hparams['n_head'])
print('核对仓库第一层，最大差值:', np.max(np.abs(x_after_block0 - reference_block0)))
assert np.allclose(x_after_block0, reference_block0)


MLP 输入: shape=(5, 768), dtype=float32
[[ 0.058 -0.062  0.175 -0.074 -0.058 -0.258]]   ← 前 1 行 × 前 6 列
升维后: shape=(5, 3072), dtype=float32
[[-0.013 -2.749 -2.69  -2.549 -0.888 -1.778]]   ← 前 1 行 × 前 6 列
GELU 后: shape=(5, 3072), dtype=float32
[[-0.007 -0.008 -0.009 -0.013 -0.166 -0.067]]   ← 前 1 行 × 前 6 列
降维后: shape=(5, 768), dtype=float32
[[ 0.034 -0.363 -0.412  0.405 -0.523 -0.398]]   ← 前 1 行 × 前 6 列
第一层最终输出: shape=(5, 768), dtype=float32
[[ 0.156 -0.795  0.394  0.341 -0.565 -0.568]]   ← 前 1 行 × 前 6 列
核对仓库第一层，最大差值: 0.0


## 11. 后续层 → 最终 logits

第一层已经拆开看过。剩余 11 层调用仓库原函数。最后做 LayerNorm，用 `wte.T` 投到整个词表。**只取最后一个位置**的 logits 来决定下一个 token。

In [12]:
x = x_after_block0
for layer_index, block in enumerate(params['blocks'][1:], start=1):
    x = transformer_block(x, **block, n_head=hparams['n_head'])
    print(f'第 {layer_index:2d} 层输出 shape: {x.shape}')
final_hidden = layer_norm(x, **params['ln_f'])
all_logits = final_hidden @ params['wte'].T
last_logits = all_logits[-1]
preview('最后的 hidden states', final_hidden, rows=1, cols=6)
print('全部位置的 logits:', all_logits.shape)
print('用于预测下一个 token 的最后一行:', last_logits.shape)


第  1 层输出 shape: (5, 768)
第  2 层输出 shape: (5, 768)
第  3 层输出 shape: (5, 768)
第  4 层输出 shape: (5, 768)
第  5 层输出 shape: (5, 768)
第  6 层输出 shape: (5, 768)
第  7 层输出 shape: (5, 768)
第  8 层输出 shape: (5, 768)
第  9 层输出 shape: (5, 768)
第 10 层输出 shape: (5, 768)
第 11 层输出 shape: (5, 768)
最后的 hidden states: shape=(5, 768), dtype=float32
[[-0.047 -0.033 -0.163 -0.007 -0.098 -0.212]]   ← 前 1 行 × 前 6 列
全部位置的 logits: (5, 50257)
用于预测下一个 token 的最后一行: (50257,)


## 12. 从 logits 选下一个 token

本仓库 `generate()` 使用 `argmax`，即 **greedy decoding**。下面显示分数最高的 8 个词片段及其 softmax 概率；这些是最终词表概率，与上面的 attention 权重不同。

In [13]:
vocab_probs = softmax(last_logits)
top_ids = np.argsort(last_logits)[-8:][::-1]
for rank, token_id in enumerate(top_ids, start=1):
    piece = encoder.decode([int(token_id)])
    print(f'{rank}. id={token_id:5d}  token={piece!r:18s}  logit={last_logits[token_id]:8.3f}  p={vocab_probs[token_id]:.4f}')
next_id = int(np.argmax(last_logits))
print('原文 + 下一 token:', repr(encoder.decode(input_ids + [next_id])))


1. id=  262  token=' the'              logit=-100.250  p=0.0846
2. id=  783  token=' now'              logit=-100.818  p=0.0479
3. id=  257  token=' a'                logit=-100.856  p=0.0462
4. id= 4881  token=' France'           logit=-101.210  p=0.0324
5. id= 6342  token=' Paris'            logit=-101.214  p=0.0322
6. id=  287  token=' in'               logit=-101.405  p=0.0266
7. id=  635  token=' also'             logit=-101.414  p=0.0264
8. id=  407  token=' not'              logit=-101.517  p=0.0238
原文 + 下一 token: 'The capital of France is the'


## 13. 与 picoGPT 完整 forward 核对

前面只是把 `gpt2.py` 的计算拆开显示，结果应该与原函数一致。然后再走一步，看生成时如何把刚得到的 token 加回输入。

In [14]:
reference_logits = gpt2(input_ids, **params, n_head=hparams['n_head'])
max_diff = np.max(np.abs(all_logits - reference_logits))
print('手动拆解 vs gpt2()，全部 logits 最大差值:', max_diff)
assert np.allclose(all_logits, reference_logits)

second_input_ids = input_ids + [next_id]
second_logits = gpt2(second_input_ids, **params, n_head=hparams['n_head'])
second_id = int(np.argmax(second_logits[-1]))
print('加入一个 token 后，输入长度:', len(second_input_ids))
print('再生成一个 token:', repr(encoder.decode([second_id])))
print('两步后的文本:', repr(encoder.decode(second_input_ids + [second_id])))


手动拆解 vs gpt2()，全部 logits 最大差值: 0.0


加入一个 token 后，输入长度: 6
再生成一个 token: ' capital'
两步后的文本: 'The capital of France is the capital'


## 回看这条链

`token ID → wte/wpe → LayerNorm → Q/K/V → QKᵀ/√d → causal mask → softmax → P·V → 多 head 合并 → 残差 → MLP → 其余层 → final LayerNorm → 词表 logits → argmax`。

可以改第 2 节的 `prompt` 后**从上到下重新运行**，观察 token 切分、attention 矩阵和最终预测如何变化。这里逐行展开的是第一层的第 0 个 head；其余计算仍使用 picoGPT 原实现。